In [1]:
import os
import numpy as np
import pandas as pd
import ast
import json

OUTPUT_DIR = "../data/output"

# Extract metadata features
def safe_len(x):
    """Safely get length of arrays"""
    if isinstance(x, (list, np.ndarray)):
        # Drop duplicated elements in the list
        return len(set(x))
    return 0

def safe_join(x):
    if isinstance(x, (list, np.ndarray)):
        if len(x) == 0:
            return ''
        return ','.join([str(i) for i in x if i is not None])
    return '' if pd.isna(x) else str(x)

def safe_json_numpy(x):
    if isinstance(x, dict):
        return json.dumps({k: v.tolist() if isinstance(v, np.ndarray) else v for k, v in x.items()})
    elif isinstance(x, np.ndarray):
        return json.dumps(x.tolist())
    elif isinstance(x, (list)):
        return json.dumps(x)
    return '' if pd.isna(x) else str(x)


### Import Data

In [2]:
review_df = pd.read_parquet(f"{OUTPUT_DIR}/review_data.parquet")
meta_df = pd.read_parquet(f"{OUTPUT_DIR}/meta_data.parquet")

### Reviews Table

In [3]:
# Add review_id as column (not index)
modified_review_df = review_df.reset_index(names="review_id")
modified_review_df["review_date"] = pd.to_datetime(modified_review_df["review_date"])

# Extract text features
modified_review_df["review_text"] = modified_review_df["review_text"].str.replace('\x00', '', regex=False)
modified_review_df["review_text_length"] = modified_review_df["review_text"].str.len()
modified_review_df["review_title_length"] = modified_review_df["review_title"].str.len()
modified_review_df["review_word_count"] = modified_review_df["review_text"].str.split().str.len()

# Sentiment proxy (simple heuristic)
modified_review_df["is_extreme_rating"] = modified_review_df["review_rating"].isin([1.0, 5.0])
modified_review_df["is_positive"] = modified_review_df["review_rating"] >= 4.0
modified_review_df["is_negative"] = modified_review_df["review_rating"] <= 2.0

# Image counts
def count_review_images(image_list):
    """Safely count number of images"""
    if isinstance(image_list, list):
        return len(image_list)
    elif pd.isna(image_list):
        return np.nan
    else:
        # If it"s not a list (maybe a string or other type), try to convert
        try:
            if isinstance(image_list, str):
                parsed = ast.literal_eval(image_list)
                return len(parsed) if isinstance(parsed, list) else 1
            else:
                return 1
        except:
            return 0

modified_review_df["num_review_img"] = modified_review_df["review_images"].apply(count_review_images)

modified_review_df["review_images"] = modified_review_df["review_images"].apply(safe_join)

# -------------------- TEMPORAL FEATURES --------------------
# Calculate days since review (from most recent date in dataset)
max_date = modified_review_df["review_date"].max()
modified_review_df["days_since_review"] = (max_date - modified_review_df["review_date"]).dt.days
modified_review_df["review_year"] = modified_review_df["review_date"].dt.year
modified_review_df["review_month"] = modified_review_df["review_date"].dt.month

# Recency weight (exponential decay with 1-year half-life)
modified_review_df["recency_weight"] = np.exp(-modified_review_df["days_since_review"] / 365.25)

modified_review_df

,review_id,asin,parent_asin,user_id,review_date,review_title,review_text,review_images,verified_purchase,helpful_vote,...,review_title_length,review_word_count,is_extreme_rating,is_positive,is_negative,num_review_img,days_since_review,review_year,review_month,recency_weight
0,0,b07bfs3g7p,b0bqsk9qcf,agci7fah4gl5fi65hylkwtmfz2cq,2019-07-03,malware,mcaffee is malware,[],False,0,...,7,3,True,False,True,0,1531,2019,7,0.015121
1,1,b00ctq6sig,b00ctq6sig,ahspldnw5oouk2plh7gxlacfbznq,2015-02-16,lots of fun,i love playing tapped out because it is fun to...,[],True,0,...,11,26,True,True,False,0,3129,2015,2,0.000190
2,2,b0066wjlu6,b0066wjlu6,ahspldnw5oouk2plh7gxlacfbznq,2013-03-04,light up the dark,i love this flashlight app! it really illumin...,[],True,0,...,17,20,True,True,False,0,3843,2013,3,0.000027
3,3,b00kcymawk,b00kcymawk,ah6catodivpvuojewhrsrcskaoha,2019-06-20,fun game,one of my favorite games,[],True,0,...,8,5,False,True,False,0,1544,2019,6,0.014593
4,4,b00p1rk566,b00p1rk566,aeiny4xoinmmjck5gz3m6mmhbn6a,2014-12-11,i am not that good at it but my kids are,cute game. i am not that good at it but my kid...,[],True,0,...,40,17,False,True,False,0,3196,2014,12,0.000158
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4880176,4880176,b0763n3mvw,b0763n3mvw,agokecumai3w6mc7a7kbfbjvpn7q,2020-12-25,gog,very fun and addictive and exciting,[],True,0,...,3,6,True,True,False,0,990,2020,12,0.066505
4880177,4880177,b00et56y48,b00et56y48,ah6amundosz2sgdidmysre4rnmdq,2021-01-06,worst game ever,worst game ever toxic people and bad connectio...,[],True,1,...,15,12,True,False,True,0,978,2021,1,0.068727
4880178,4880178,b009zkspdk,b009zkspdk,ah4pj73qn75ajm5vsct53aoadcga,2013-05-28,better!!!,this fabulous game is 10000 times better than ...,[],True,2,...,9,36,True,True,False,0,3758,2013,5,0.000034
4880179,4880179,b00m9j3ioa,b00m9j3ioa,ahqetdykhvdngmgbwvjl6vjxjgfq,2016-09-03,it has everything i need and more,awesome! i upgraded from coreldraw 8. i was wo...,[],True,0,...,33,20,True,True,False,0,2564,2016,9,0.000894


In [4]:
modified_review_df.dtypes

review_id                      int64
asin                             str
parent_asin                      str
user_id                          str
review_date            datetime64[s]
review_title                     str
review_text                      str
review_images                    str
verified_purchase               bool
helpful_vote                   int64
review_rating                float64
review_text_length             int64
review_title_length            int64
review_word_count              int64
is_extreme_rating               bool
is_positive                     bool
is_negative                     bool
num_review_img                 int64
days_since_review              int64
review_year                    int32
review_month                   int32
recency_weight               float64
dtype: object

### User Table

In [5]:
user_df = (
    modified_review_df
    .merge(
        meta_df[["parent_asin", "price"]],
        how="left",
        on="parent_asin"
    )
    .assign(
        is_free = lambda row: row["price"] == 0.0
    )
    .groupby("user_id")
    .agg({
        # Basic stats
        "review_id": "count",
        "review_rating": ["mean", "std", "min", "max"],

        # Temporal
        "review_date": ["min", "max"],
        "recency_weight": "sum",

        # Quality signals
        "helpful_vote": ["sum", "mean"],
        "verified_purchase": "sum",

        # Text engagement
        "review_text_length": "mean",
        "review_word_count": "mean",
        "num_review_img": "sum",

        # Rating patterns
        "is_extreme_rating": "mean",
        "is_positive": "mean",
        "is_negative": "mean",

        # Price sensitivity
        "price": "mean",
        "is_free": "mean"
    })
    .reset_index()
)

# Flatten multi-level columns
user_df.columns = ["_".join(col).strip("_") for col in user_df.columns]

# Rename for clarity
user_df = user_df.rename(columns={
    "review_id_count": "num_reviews",
    "review_rating_mean": "avg_rating_given",
    "review_rating_std": "rating_std",
    "review_rating_min": "min_rating_given",
    "review_rating_max": "max_rating_given",
    "review_date_min": "first_review_date",
    "review_date_max": "last_review_date",
    "recency_weight_sum": "total_recency_weight",
    "helpful_vote_sum": "total_helpful_votes_received",
    "helpful_vote_mean": "avg_helpful_votes_per_review",
    "verified_purchase_sum": "num_verified_purchases",
    "review_text_length_mean": "avg_review_length",
    "review_word_count_mean": "avg_review_words",
    "num_review_images_sum": "total_review_images",
    "is_extreme_rating_mean": "extreme_rating_ratio",
    "is_positive_mean": "positive_rating_ratio",
    "is_negative_mean": "negative_rating_ratio",
    "price_mean": "avg_price_purchased",
    "is_free_mean": "free_app_ratio"
})

# Derive additional features
user_df["days_active"] = (
    user_df["last_review_date"] - user_df["first_review_date"]
).dt.days + 1

user_df["reviews_per_day"] = (
    user_df["num_reviews"] / user_df["days_active"]
)

user_df["verified_purchase_ratio"] = (
    user_df["num_verified_purchases"] / user_df["num_reviews"]
)

# User segmentation (from EDA)
user_df["user_segment"] = pd.cut(
    user_df["num_reviews"],
    bins=[0, 1, 10, np.inf],
    labels=["one_time", "occasional", "power_user"]
)

# Rating discriminativeness (higher std = more discriminating)
user_df["is_discriminating"] = user_df["rating_std"] > 1.0

print(f"Total users: {len(user_df)}")
print(f"User segments:\n{user_df['user_segment'].value_counts()}")

user_df

Total users: 2589466
User segments:
user_segment
one_time      1743452
occasional     814338
power_user      31676
Name: count, dtype: int64


,user_id,num_reviews,avg_rating_given,rating_std,min_rating_given,max_rating_given,first_review_date,last_review_date,total_recency_weight,total_helpful_votes_received,...,extreme_rating_ratio,positive_rating_ratio,negative_rating_ratio,avg_price_purchased,free_app_ratio,days_active,reviews_per_day,verified_purchase_ratio,user_segment,is_discriminating
0,ae2222frpdmnomyomcwiantxp7uq,1,5.000000,NaN,5.0,5.0,2014-07-06,2014-07-06,0.000103,0,...,1.000000,1.000000,0.000000,NaN,0.000000,1,1.000000,1.0,one_time,False
1,ae22232ob6s7uac75jufyrgbshiq,1,5.000000,NaN,5.0,5.0,2020-05-16,2020-05-16,0.036116,1,...,1.000000,1.000000,0.000000,0.000000,1.000000,1,1.000000,1.0,one_time,False
2,ae22236afrrsmqikgg7tptb75qea,9,3.444444,1.878238,1.0,5.0,2012-11-22,2018-12-15,0.009146,79,...,0.777778,0.666667,0.333333,1.328333,0.333333,2215,0.004063,1.0,occasional,True
3,ae2224fsuk5av5r2usyxinuntw7q,3,2.333333,2.309401,1.0,5.0,2015-03-10,2015-03-24,0.000614,0,...,1.000000,0.333333,0.666667,0.000000,1.000000,15,0.200000,1.0,occasional,True
4,ae2226penzttcdkfgrtucux2nu2q,2,3.000000,2.828427,1.0,5.0,2021-05-13,2021-05-13,0.194609,3,...,1.000000,0.500000,0.500000,0.000000,1.000000,1,2.000000,1.0,occasional,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2589461,ahzzzngyuwv64yjbzgd3uc5y3liq,2,5.000000,0.000000,5.0,5.0,2015-01-15,2015-01-15,0.000349,1,...,1.000000,1.000000,0.000000,1.990000,0.000000,1,2.000000,1.0,occasional,False
2589462,ahzzzuba3pttfi67ios2mftjavea,1,5.000000,NaN,5.0,5.0,2014-12-31,2014-12-31,0.000167,1,...,1.000000,1.000000,0.000000,0.000000,1.000000,1,1.000000,1.0,one_time,False
2589463,ahzzzwvhlytqu55pd4fjuluektxa,2,5.000000,0.000000,5.0,5.0,2014-04-12,2017-07-14,0.002193,2,...,1.000000,1.000000,0.000000,NaN,0.000000,1190,0.001681,1.0,occasional,False
2589464,ahzzzy4dflawpbqyfqfwvacngura,4,4.500000,0.577350,4.0,5.0,2016-06-03,2016-08-19,0.003077,37,...,0.500000,1.000000,0.000000,0.247500,0.750000,78,0.051282,1.0,occasional,False


In [6]:
user_df.dtypes

user_id                                   str
num_reviews                             int64
avg_rating_given                      float64
rating_std                            float64
min_rating_given                      float64
max_rating_given                      float64
first_review_date               datetime64[s]
last_review_date                datetime64[s]
total_recency_weight                  float64
total_helpful_votes_received            int64
avg_helpful_votes_per_review          float64
num_verified_purchases                  int64
avg_review_length                     float64
avg_review_words                      float64
num_review_img_sum                      int64
extreme_rating_ratio                  float64
positive_rating_ratio                 float64
negative_rating_ratio                 float64
avg_price_purchased                   float64
free_app_ratio                        float64
days_active                             int64
reviews_per_day                   

### Items Table

In [7]:
item_df = meta_df.copy()

item_df["num_item_img"] = item_df["item_images"].apply(
    lambda d: (
        sum(bool(x) for x in d.get("hi_res", [])) +
        sum(bool(x) for x in d.get("large", [])) +
        sum(bool(x) for x in d.get("thumb", []))
    )
)

item_df["num_item_videos"] = item_df["item_videos"].apply(
    lambda d: len(d.get("url", []))
)

# Handle price properly
item_df["price"] = item_df["price"].fillna(-1)  # -1 indicates unknown
item_df["is_free"] = item_df["price"] == 0.0
item_df["has_price_info"] = item_df["price"] >= 0
item_df["price_bucket"] = pd.cut(
    item_df["price"],
    bins=[-1, 0, 1, 10, 25, 50, 100, float("inf")],
    labels=["unknown", "0-1", "1-10", "10-25", "25-50", "50-100", "100+"]
)

item_df["num_categories"] = item_df["categories"].apply(safe_len)

item_df["categories"] = item_df["categories"].apply(safe_join)

item_df["description"] = item_df["description"].apply(safe_join)
item_df["features"] = item_df["features"].apply(safe_join)
item_df["details"] = item_df["details"].apply(safe_join)

item_df["item_images"] = item_df["item_images"].apply(safe_json_numpy)
item_df["item_videos"] = item_df["item_videos"].apply(safe_json_numpy)

item_df

,parent_asin,item_title,main_category,categories,description,features,details,item_images,item_videos,item_rating,store,price,num_item_img,num_item_videos,is_free,has_price_info,price_bucket,num_categories
0,b00vrpsgeo,accupressure guide,appstore for android,,Acupressure technique is very ancient and very...,All the pressing point has been explained with...,"{""Release Date"": ""2015"", ""Date first listed on...","{""hi_res"": [null, null, null, null], ""large"": ...","{""title"": [], ""url"": [], ""user_id"": []}",NaN,mappsguru,0.00,4,0,True,True,unknown,0
1,b00nwqxxhq,ankylosaurus fights back - smithsonian's prehi...,appstore for android,,Join Ankylosaurus in this interactive book app...,ENCOURAGE literacy skills with highlighted nar...,"{""Release Date"": ""2014"", ""Date first listed on...","{""hi_res"": [null, null, null, null, null], ""la...","{""title"": [], ""url"": [], ""user_id"": []}",NaN,"oceanhouse media, inc",2.99,5,0,False,True,1-10,0
2,b00rfkp6ac,mahjong 2015,appstore for android,,Mahjong 2015 is a free solitaire matching game...,Mahjong 2015 is a free solitaire matching game...,"{""Release Date"": ""2014"", ""Date first listed on...","{""hi_res"": [null, null, null], ""large"": [""http...","{""title"": [], ""url"": [], ""user_id"": []}",NaN,sophiathach,0.00,3,0,True,True,unknown,0
3,b00sp2qu0e,jewels brick breakout,appstore for android,,Jewels Brick Breakout is a glowing jewels bric...,"Game Features:,- Intuitive touch controls with...","{""Release Date"": ""2015"", ""Date first listed on...","{""hi_res"": [null, null, null, null, null, null...","{""title"": [], ""url"": [], ""user_id"": []}",NaN,bad chicken,0.00,9,0,True,True,unknown,0
4,b01dzit64o,traffic police: off-road cub,appstore for android,,Become the best road police officer in Cube Ci...,"In this game you will find:,- Killer police ca...","{""Release Date"": ""2016"", ""Date first listed on...","{""hi_res"": [null, null, null, null], ""large"": ...","{""title"": [], ""url"": [], ""user_id"": []}",NaN,dast 2 for metro,0.00,4,0,True,True,unknown,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
89246,b006kwvrgi,how to start a commercial loan broker plus bus...,software,"business & marketing plans,business & office,b...",The How to Start a Commercial Loan Brokerage G...,Everything You Need to Know About Starting a C...,"{""Is Discontinued By Manufacturer"": ""No"", ""Ite...","{""hi_res"": [null, null, null, null, null], ""la...","{""title"": [], ""url"": [], ""user_id"": []}",2.0,howtostartabusinessdb,29.95,10,0,False,True,25-50,4
89247,b000j54pra,norton uninstall deluxe,software,"software,utilities",The safe uninstaller from the makers of Norton...,Safely removes programs and files you no longe...,"{""Is Discontinued By Manufacturer"": ""No"", ""Pac...","{""hi_res"": [null], ""large"": [""https://m.media-...","{""title"": [], ""url"": [], ""user_id"": []}",2.0,symantec,-1.00,2,0,False,False,NaN,2
89248,b00mg0ohb0,beginning pro tools le 8 [instant access],NaN,"education & reference,software",This video takes you from start to finish on a...,,"{""Language"": ""English"", ""Item model number"": ""...","{""hi_res"": [null], ""large"": [""https://m.media-...","{""title"": [], ""url"": [], ""user_id"": []}",3.0,alfred music,15.95,2,0,False,True,10-25,2
89249,b001gn6yuk,tell me more spanish performance version 9 (2 ...,NaN,"education & reference,languages,software","From the Manufacturer,This award-winning langu...",Spanish language-learning software with 2 leve...,"{""Language"": ""English, French"", ""Package Dimen...","{""hi_res"": [null], ""large"": [""https://m.media-...","{""title"": [], ""url"": [], ""user_id"": []}",2.0,auralog,209.00,2,0,False,True,100+,3


In [8]:
item_popularity_df = (
    modified_review_df
    .groupby("parent_asin")
    .agg({
        # Basic stats
        "user_id": "count",
        "review_rating": ["mean", "std", "min", "max"],

        # Temporal
        "review_date": ["min", "max"],
        "recency_weight": "sum",

        # Quality signals
        "helpful_vote": ["sum", "mean"],
        "verified_purchase": "sum",

        # Rating patterns
        "is_positive": "mean",
        "is_negative": "mean",
    })
    .reset_index()
)

# Flatten columns
item_popularity_df.columns = ["_".join(col).strip("_") for col in item_popularity_df.columns]

# Rename
item_popularity_df = item_popularity_df.rename(columns={
    "user_id_count": "num_reviews",
    "review_rating_mean": "avg_rating",
    "review_rating_std": "rating_std",
    "review_rating_min": "min_rating",
    "review_rating_max": "max_rating",
    "review_date_min": "first_review_date",
    "review_date_max": "last_review_date",
    "recency_weight_sum": "total_recency_weight",
    "helpful_vote_sum": "total_helpful_votes",
    "helpful_vote_mean": "avg_helpful_votes",
    "helpful_vote_max": "max_helpful_votes",
    "verified_purchase_sum": "num_verified_reviews",
    "is_positive_mean": "positive_review_ratio",
    "is_negative_mean": "negative_review_ratio"
})

item_popularity_df

,parent_asin,num_reviews,avg_rating,rating_std,min_rating,max_rating,first_review_date,last_review_date,total_recency_weight,total_helpful_votes,avg_helpful_votes,num_verified_reviews,positive_review_ratio,negative_review_ratio
0,0005162092,2,5.000000,0.000000,5.0,5.0,2013-06-08,2015-02-02,2.182259e-04,0,0.000000,2,1.000000,0.000000
1,0028217012,1,1.000000,NaN,1.0,1.0,2012-10-13,2012-10-13,1.826844e-05,0,0.000000,1,0.000000,1.000000
2,0030369371,1,3.000000,NaN,3.0,3.0,2016-02-21,2016-02-21,5.241512e-04,0,0.000000,1,0.000000,0.000000
3,0071480935,4,4.250000,0.957427,3.0,5.0,2007-08-15,2014-08-29,1.869636e-04,5,1.250000,4,0.750000,0.000000
4,0072119527,1,5.000000,NaN,5.0,5.0,2000-03-25,2000-03-25,6.456290e-11,8,8.000000,0,1.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
89241,b0c9bbdbpm,170,3.117647,1.813244,1.0,5.0,2019-05-10,2023-08-11,3.839281e+01,278,1.635294,156,0.517647,0.435294
89242,b0c9v8pzny,1,1.000000,NaN,1.0,1.0,2023-08-21,2023-08-21,9.441267e-01,0,0.000000,0,0.000000,1.000000
89243,b0cbn6ygz3,2,4.000000,1.414214,3.0,5.0,2019-02-01,2019-07-14,2.555735e-02,1,0.500000,2,0.500000,0.000000
89244,b0cc7jglms,8,4.875000,0.353553,4.0,5.0,2023-03-01,2023-05-17,5.013251e+00,4,0.500000,8,1.000000,0.000000


In [9]:
item_df = item_df.merge(
    item_popularity_df,
    how="left",
    on="parent_asin"
)

# Derive features
item_df["days_on_platform"] = (
    item_df["last_review_date"] - item_df["first_review_date"]
).dt.days + 1

item_df["reviews_per_day"] = (
    item_df["num_reviews"] / item_df["days_on_platform"]
)

item_df["verified_review_ratio"] = (
    item_df["num_verified_reviews"] / item_df["num_reviews"]
)

# Item popularity segment (from EDA)
item_df["popularity_segment"] = pd.cut(
    item_df["num_reviews"],
    bins=[0, 1, 10, 100, np.inf],
    labels=["cold_start", "low_coverage", "medium", "popular"]
)

# Quality score (Wilson lower bound for rating confidence)
def wilson_lower_bound(pos, n, confidence=0.95):
    """Wilson score interval for rating confidence"""
    if n == 0:
        return 0
    z = 1.96  # 95% confidence
    phat = pos / n
    return (phat + z*z/(2*n) - z * np.sqrt((phat*(1-phat)+z*z/(4*n))/n))/(1+z*z/n)

item_df["quality_score"] = item_df.apply(
    lambda row: wilson_lower_bound(
        row["positive_review_ratio"] * row["num_reviews"],
        row["num_reviews"]
    ),
    axis=1
)

print(f"Total items: {len(item_df)}")
print(f"Popularity segments:\n{item_df['popularity_segment'].value_counts()}")

item_df

Total items: 89251
Popularity segments:
popularity_segment
low_coverage    43085
cold_start      24636
medium          16603
popular          4922
Name: count, dtype: int64


,parent_asin,item_title,main_category,categories,description,features,details,item_images,item_videos,item_rating,...,total_helpful_votes,avg_helpful_votes,num_verified_reviews,positive_review_ratio,negative_review_ratio,days_on_platform,reviews_per_day,verified_review_ratio,popularity_segment,quality_score
0,b00vrpsgeo,accupressure guide,appstore for android,,Acupressure technique is very ancient and very...,All the pressing point has been explained with...,"{""Release Date"": ""2015"", ""Date first listed on...","{""hi_res"": [null, null, null, null], ""large"": ...","{""title"": [], ""url"": [], ""user_id"": []}",NaN,...,3.0,1.000000,3.0,1.000000,0.000000,250.0,0.012000,1.000000,low_coverage,0.438494
1,b00nwqxxhq,ankylosaurus fights back - smithsonian's prehi...,appstore for android,,Join Ankylosaurus in this interactive book app...,ENCOURAGE literacy skills with highlighted nar...,"{""Release Date"": ""2014"", ""Date first listed on...","{""hi_res"": [null, null, null, null, null], ""la...","{""title"": [], ""url"": [], ""user_id"": []}",NaN,...,0.0,0.000000,1.0,1.000000,0.000000,29.0,0.068966,0.500000,low_coverage,0.342372
2,b00rfkp6ac,mahjong 2015,appstore for android,,Mahjong 2015 is a free solitaire matching game...,Mahjong 2015 is a free solitaire matching game...,"{""Release Date"": ""2014"", ""Date first listed on...","{""hi_res"": [null, null, null], ""large"": [""http...","{""title"": [], ""url"": [], ""user_id"": []}",NaN,...,34.0,1.172414,29.0,0.551724,0.137931,1018.0,0.028487,1.000000,medium,0.375477
3,b00sp2qu0e,jewels brick breakout,appstore for android,,Jewels Brick Breakout is a glowing jewels bric...,"Game Features:,- Intuitive touch controls with...","{""Release Date"": ""2015"", ""Date first listed on...","{""hi_res"": [null, null, null, null, null, null...","{""title"": [], ""url"": [], ""user_id"": []}",NaN,...,0.0,0.000000,2.0,0.500000,0.500000,37.0,0.054054,1.000000,low_coverage,0.094529
4,b01dzit64o,traffic police: off-road cub,appstore for android,,Become the best road police officer in Cube Ci...,"In this game you will find:,- Killer police ca...","{""Release Date"": ""2016"", ""Date first listed on...","{""hi_res"": [null, null, null, null], ""large"": ...","{""title"": [], ""url"": [], ""user_id"": []}",NaN,...,23.0,7.666667,3.0,0.333333,0.666667,458.0,0.006550,1.000000,low_coverage,0.061490
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
89246,b006kwvrgi,how to start a commercial loan broker plus bus...,software,"business & marketing plans,business & office,b...",The How to Start a Commercial Loan Brokerage G...,Everything You Need to Know About Starting a C...,"{""Is Discontinued By Manufacturer"": ""No"", ""Ite...","{""hi_res"": [null, null, null, null, null], ""la...","{""title"": [], ""url"": [], ""user_id"": []}",2.0,...,0.0,0.000000,1.0,0.000000,1.000000,1.0,1.000000,1.000000,cold_start,0.000000
89247,b000j54pra,norton uninstall deluxe,software,"software,utilities",The safe uninstaller from the makers of Norton...,Safely removes programs and files you no longe...,"{""Is Discontinued By Manufacturer"": ""No"", ""Pac...","{""hi_res"": [null], ""large"": [""https://m.media-...","{""title"": [], ""url"": [], ""user_id"": []}",2.0,...,0.0,0.000000,0.0,0.000000,1.000000,1.0,1.000000,0.000000,cold_start,0.000000
89248,b00mg0ohb0,beginning pro tools le 8 [instant access],NaN,"education & reference,software",This video takes you from start to finish on a...,,"{""Language"": ""English"", ""Item model number"": ""...","{""hi_res"": [null], ""large"": [""https://m.media-...","{""title"": [], ""url"": [], ""user_id"": []}",3.0,...,5.0,1.666667,2.0,0.333333,0.666667,481.0,0.006237,0.666667,low_coverage,0.061490
89249,b001gn6yuk,tell me more spanish performance version 9 (2 ...,NaN,"education & reference,languages,software","From the Manufacturer,This award-winning langu...",Spanish language-learning software with 2 leve...,"{""Language"": ""English, French"", ""P

In [10]:
item_df.dtypes

parent_asin                        str
item_title                         str
main_category                      str
categories                         str
description                        str
features                           str
details                            str
item_images                        str
item_videos                        str
item_rating                    float64
store                              str
price                          float64
num_item_img                     int64
num_item_videos                  int64
is_free                           bool
has_price_info                    bool
price_bucket                  category
num_categories                   int64
num_reviews                    float64
avg_rating                     float64
rating_std                     float64
min_rating                     float64
max_rating                     float64
first_review_date        datetime64[s]
last_review_date         datetime64[s]
total_recency_weight     

### Export Data

In [11]:
# Create output directory if it doesn't exist
os.makedirs(OUTPUT_DIR, exist_ok=True)

modified_review_df.to_parquet(f"{OUTPUT_DIR}/user-item-interaction.parquet", index=False)
user_df.to_parquet(f"{OUTPUT_DIR}/user.parquet", index=False)
item_df.to_parquet(f"{OUTPUT_DIR}/item.parquet", index=False)

print("Complete")

Complete


In [12]:
modified_review_df.to_csv(f"{OUTPUT_DIR}/user-item-interaction.csv", index=False)
user_df.to_csv(f"{OUTPUT_DIR}/user.csv", index=False)
item_df.to_csv(f"{OUTPUT_DIR}/item.csv", index=False)